In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

train_w = pd.read_parquet("../data/processed/train_woe.parquet")
test_w  = pd.read_parquet("../data/processed/test_woe.parquet")

# Same model as in notebook 04
model = LogisticRegression(C=10, max_iter=1000).fit(
    train_w.drop(columns="target"), train_w["target"])
pd_test = model.predict_proba(test_w.drop(columns="target"))[:, 1]

# Loan amounts of the 2015 test portfolio (same row order as test_woe)
clean = pd.read_parquet("../data/processed/accepted_clean.parquet",
                        columns=["issue_d", "loan_amnt"])
test_amt = clean[(clean["issue_d"] >= "2015-01-01") &
                 (clean["issue_d"] < "2016-01-01")]["loan_amnt"].values
assert len(test_amt) == len(pd_test), "row count mismatch"
print(len(pd_test), "loans in the 2015 test portfolio")

283026 loans in the 2015 test portfolio


In [2]:
# Same file as in notebook 01 (point to the .gz file if the csv was deleted)
RAW = "../data/raw/accepted_2007_to_2018Q4.csv"
cols = ["issue_d", "term", "loan_status", "loan_amnt",
        "total_rec_prncp", "recoveries", "collection_recovery_fee"]

# Read in chunks; keep finished 36-month loans only
parts = []
for ch in pd.read_csv(RAW, usecols=cols, chunksize=200_000, low_memory=False):
    ch = ch[ch["loan_status"].isin(["Fully Paid", "Charged Off"])]
    parts.append(ch[ch["term"].str.strip() == "36 months"])

raw = pd.concat(parts, ignore_index=True)
raw["issue_d"] = pd.to_datetime(raw["issue_d"], format="%b-%Y")
raw["bad"] = (raw["loan_status"] == "Charged Off").astype(int)

# Exposure left at default, and recoveries net of collection fees
raw["ead_at_default"] = raw["loan_amnt"] - raw["total_rec_prncp"]
raw["net_recovery"]   = raw["recoveries"] - raw["collection_recovery_fee"]
raw["loss"] = np.where(raw["bad"] == 1,
                       raw["ead_at_default"] - raw["net_recovery"], 0)

train_raw = raw[(raw["issue_d"] >= "2012-01-01") & (raw["issue_d"] < "2015-01-01")]
test_raw  = raw[(raw["issue_d"] >= "2015-01-01") & (raw["issue_d"] < "2016-01-01")]

# Parameters are estimated on the training period only (no look-ahead)
d = train_raw[train_raw["bad"] == 1]
LGD = d["loss"].sum() / d["ead_at_default"].sum()               # loss as a share of exposure at default
EAD_RATIO = d["ead_at_default"].sum() / d["loan_amnt"].sum()    # share of the loan still outstanding at default
print(f"LGD = {LGD:.3f}   EAD ratio = {EAD_RATIO:.3f}")

LGD = 0.889   EAD ratio = 0.574


In [3]:
# Loan-level ECL = PD x LGD x EAD
ecl = pd_test * LGD * (EAD_RATIO * test_amt)

pred_rate = ecl.sum() / test_amt.sum()
real_rate = test_raw["loss"].sum() / test_raw["loan_amnt"].sum()

print(f"Predicted ECL rate : {pred_rate:.4f}")
print(f"Realised loss rate : {real_rate:.4f}")
print(f"Ratio (realised / predicted): {real_rate / pred_rate:.2f}")

Predicted ECL rate : 0.0637
Realised loss rate : 0.0756
Ratio (realised / predicted): 1.19


In [4]:
tbl = pd.DataFrame({"pd": pd_test, "ecl": ecl, "amt": test_amt})
tbl["decile"] = pd.qcut(tbl["pd"], 10, labels=False, duplicates="drop") + 1

out = tbl.groupby("decile").agg(avg_pd=("pd", "mean"),
                                ecl=("ecl", "sum"),
                                exposure=("amt", "sum"))
out["ecl_rate"] = out["ecl"] / out["exposure"]
out.round(4)

,avg_pd,ecl,exposure,ecl_rate
decile,,,,
1,0.0426,9.449725e+06,435617975.0,0.0217
2,0.0672,1.421266e+07,415151075.0,0.0342
3,0.0844,1.689482e+07,392816025.0,0.0430
4,0.0995,1.878851e+07,370592250.0,0.0507
5,0.1143,2.095385e+07,359511050.0,0.0583
6,0.1301,2.312217e+07,348674575.0,0.0663
7,0.1475,2.555819e+07,339696150.0,0.0752
8,0.1688,2.836706e+07,329601500.0,0.0861
9,0.1979,3.267138e+07,323950575.0,0.1009
